In [16]:
import os
import uuid
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb


In [17]:
BASE_DIR = "wiki_en_1000"
os.makedirs(BASE_DIR, exist_ok=True)

In [18]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []

    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)

    return chunks


In [19]:
model = SentenceTransformer("all-MiniLM-L6-v2")


In [20]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(
    Settings(
        persist_directory="./chroma_store",
        anonymized_telemetry=False
    )
)

collection = client.get_or_create_collection(
    name="wiki_hf_1000"
)


In [21]:
import math

MAX_CHUNKS_PER_DOC = 20
EMBED_BATCH_SIZE = 16 

for filename in os.listdir(BASE_DIR):
    file_path = os.path.join(BASE_DIR, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = chunk_text(text)
    chunks = chunks[:MAX_CHUNKS_PER_DOC]

    if not chunks:
        continue

    # Embed in batches
    for i in range(0, len(chunks), EMBED_BATCH_SIZE):
        batch_chunks = chunks[i:i + EMBED_BATCH_SIZE]
        embeddings = model.encode(batch_chunks, show_progress_bar=False)

        collection.add(
            documents=batch_chunks,
            embeddings=[e.tolist() for e in embeddings],
            ids=[str(uuid.uuid4()) for _ in batch_chunks],
            metadatas=[{
                "article_id": filename.replace(".txt", ""),
                "source_file": filename
            } for _ in batch_chunks]
        )

print("Embedding completed")


Embedding completed


In [22]:
query = "What is machine learning?"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=5
)


In [23]:
for i, meta in enumerate(results["metadatas"][0]):
    print(f"\nResult {i+1}")
    print("Article ID:", meta["article_id"])



Result 1
Article ID: 50447852

Result 2
Article ID: 50447852

Result 3
Article ID: 48892446

Result 4
Article ID: 48892446

Result 5
Article ID: 48892446


# Queries

In [24]:
import json

with open("queries_1000.json", "r", encoding="utf-8") as f:
    loaded_queries = json.load(f)

print(len(loaded_queries))
print(loaded_queries[:5])


1000
['when is the last episode of season 8 of the walking dead', 'in greek mythology who was the goddess of spring growth', 'benefits of colonial life for single celled organisms', 'how many season of the man in the high castle', 'who was the first ministry head of state in nigeria']


In [25]:
query_embeddings = model.encode(
    loaded_queries,
    batch_size=8,
    convert_to_numpy=True,
    show_progress_bar=True
).tolist()

results = collection.query(
    query_embeddings=query_embeddings,
    n_results=3
)


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

In [26]:
import time
from collections import defaultdict
import random

def run_queries_metrics(
    collection,
    model,
    queries,
    k=3,
    batch_size=8,
    num_examples=5
):

    query_latencies = []
    per_query_results = defaultdict(list)

    total_start = time.perf_counter()

    
    embeddings = model.encode(
        queries,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=False
    ).tolist()

    for q, emb in zip(queries, embeddings):
        q_start = time.perf_counter()

        res = collection.query(
            query_embeddings=[emb],
            n_results=k,
            include=["documents", "metadatas"]
        )

        q_time = time.perf_counter() - q_start
        query_latencies.append(q_time)

        docs = res["documents"][0]
        metas = res["metadatas"][0]

        for rank, (doc, meta) in enumerate(zip(docs, metas), 1):
            per_query_results[q].append({
                "rank": rank,
                "article_id": meta.get("article_id"),
                "source_file": meta.get("source_file"),
                "text": doc
            })

    total_time = time.perf_counter() - total_start
    avg_latency = sum(query_latencies) / len(query_latencies)
    throughput = len(queries) / total_time if total_time > 0 else 0

        
    print("\n=== QUERY EVALUATION SUMMARY ===")
    print(f"Total queries        : {len(queries)}")
    print(f"Total time (s)       : {total_time:.2f}")
    print(f"Avg latency (s)      : {avg_latency:.4f}")
    print(f"Throughput (q/s)     : {throughput:.2f}")

    
    print("\n=== SAMPLE QUERY RESULTS ===")

    example_queries = random.sample(
        queries, min(num_examples, len(queries))
    )

    for idx, q in enumerate(example_queries, 1):
        print(f"\nExample {idx}")
        print(f"Query: {q}")

        for res in per_query_results[q]:
            print(f"  → Rank {res['rank']}")
            print(f"    Article ID : {res['article_id']}")
            # print(f"    Source     : {res['source_file']}")
            print(f"    Text       : {res['text'][:150]}...")

    return {
        "queries": len(queries),
        "total_time": total_time,
        "avg_latency": avg_latency,
        "throughput": throughput,
        "results": per_query_results
    }


In [27]:
import json

with open("queries_1000.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

metrics = run_queries_metrics(
    collection=collection,
    model=model,
    queries=queries,
    k=3,
    batch_size=8,
    num_examples=5
)



=== QUERY EVALUATION SUMMARY ===
Total queries        : 1000
Total time (s)       : 10.68
Avg latency (s)      : 0.0018
Throughput (q/s)     : 93.66

=== SAMPLE QUERY RESULTS ===

Example 1
Query: pulmonary congestion is when the blood from the ventricle flows back into the
  → Rank 1
    Article ID : 56275513
    Text       : TITLE: Ventadour River The Ventadour River is a tributary of the south shore of Robert Lake flowing into Eeyou Istchee Baie-James (municipality), in J...
  → Rank 2
    Article ID : 56275513
    Text       : TITLE: Ventadour River The Ventadour River is a tributary of the south shore of Robert Lake flowing into Eeyou Istchee Baie-James (municipality), in J...
  → Rank 3
    Article ID : 403899
    Text       : had been splashed, the work of unloading the transports and cargo ships resumed. One "Betty", crippled by antiaircraft fire, crashed into the after su...

Example 2
Query: who played daniel in the bible mini series
  → Rank 1
    Article ID : 52346
    Tex